In [1]:
import os
import glob
import pandas as pd

# CRUCE ENTRE IMPREVISTAS  VS EVENLIST.

In [2]:
import pandas as pd
import os
from datetime import datetime

# Cargar las bases de datos
df1 = pd.read_excel(r'C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\IMPUT-TOTAL\IMPREVISTAS-TOTAL.xlsx')
df2 = pd.read_excel(r'C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\GEMAS-RETIN\IMPUT-TOTAL\EventList.xlsx')

# Verificar las columnas disponibles
print("Columnas en df1:", df1.columns.tolist())
print("Columnas en df2:", df2.columns.tolist())

df2.rename(columns={
    'Fecha inicio/hora': 'Fecha inicio-hora',
}, inplace=True)

# Asegurarse de que las columnas de fecha estén en formato datetime
def convertir_fechas(df):
    df['Fecha inicio-hora'] = pd.to_datetime(df['Fecha inicio-hora'], dayfirst=True, errors='coerce')
    df['Fecha Fin'] = pd.to_datetime(df['Fecha Fin'], dayfirst=True, errors='coerce')
    return df



display(df1)
display(df2)
df1 = convertir_fechas(df1)
df2 = convertir_fechas(df2)




# Crear directorio para resultados
output_dir = 'Resultados_Cruce_BD'
os.makedirs(output_dir, exist_ok=True)

# Función para exportar a Excel con formato
def exportar_excel(df, filename, sheet_name='Resultados'):
    """Exportar DataFrame a Excel con formato mejorado"""
    try:
        with pd.ExcelWriter(os.path.join(output_dir, filename), engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Ajustar ancho de columnas
            worksheet = writer.sheets[sheet_name]
            for idx, col in enumerate(df.columns):
                max_length = max(df[col].astype(str).str.len().max(), len(col))
                worksheet.column_dimensions[chr(65 + idx)].width = min(max_length + 2, 50)
        
        print(f"✓ Archivo exportado: {filename}")
        return True
    except Exception as e:
        print(f"✗ Error exportando {filename}: {e}")
        return False

# 1. CRUCE EXACTO - Coincidencias perfectas en las 3 columnas
print("\n" + "="*60)
print("1. BUSCANDO COINCIDENCIAS EXACTAS")
print("="*60)

df_exactas = pd.merge(
    df1, 
    df2, 
    on=['LMT', 'Fecha inicio-hora', 'Fecha Fin'], 
    how='inner',
    suffixes=('_IMPREVISTAS', '_EVENTLIST')
)

if len(df_exactas) > 0:
    print(f"✓ Coincidencias exactas encontradas: {len(df_exactas)} registros")
    
    # Exportar coincidencias exactas
    exportar_excel(df_exactas, '01_Coincidencias_Exactas.xlsx')
    
    # Resumen de coincidencias exactas
    resumen_exactas = pd.DataFrame({
        'Tipo': ['Coincidencias Exactas'],
        'Cantidad_Registros': [len(df_exactas)],
        'LMT_Encontrados': [df_exactas['LMT'].nunique()],
        'Fecha_Mas_Antigua': [df_exactas['Fecha inicio-hora'].min()],
        'Fecha_Mas_Reciente': [df_exactas['Fecha inicio-hora'].max()]
    })
    
else:
    print("✗ No se encontraron coincidencias exactas")
    df_exactas = pd.DataFrame()  # DataFrame vacío
    resumen_exactas = pd.DataFrame({
        'Tipo': ['Coincidencias Exactas'],
        'Cantidad_Registros': [0],
        'LMT_Encontrados': [0],
        'Fecha_Mas_Antigua': ['N/A'],
        'Fecha_Mas_Reciente': ['N/A']
    })

# 2. CRUCE APROXIMADO - Mismo LMT con diferencias de tiempo
print("\n" + "="*60)
print("2. BUSCANDO COINCIDENCIAS APROXIMADAS")
print("="*60)

resultados_aproximados = []
lmt_procesados = set()

for idx1, row1 in df1.iterrows():
    lmt = row1['LMT']
    if lmt in lmt_procesados:
        continue
        
    coincidencias_lmt = df2[df2['LMT'] == lmt]
    
    if not coincidencias_lmt.empty:
        lmt_procesados.add(lmt)
        
        for idx2, row2 in coincidencias_lmt.iterrows():
            # Calcular diferencias en minutos
            diff_inicio = abs((row1['Fecha inicio-hora'] - row2['Fecha inicio-hora']).total_seconds() / 60)
            diff_fin = abs((row1['Fecha Fin'] - row2['Fecha Fin']).total_seconds() / 60)
            
            # Determinar si la diferencia es observable
            diferencia_observable = "Observable" if (diff_inicio > 5 or diff_fin > 5) else "Permitido"
            
            # Obtener referencia si existe
            referencia = row2['Referencia'] if 'Referencia' in row2 else 'N/A'
            
            resultados_aproximados.append({
                'LMT': lmt,
                'Fecha_Inicio_IMPREVISTAS': row1['Fecha inicio-hora'],
                'Fecha_Inicio_EVENTLIST': row2['Fecha inicio-hora'],
                'Fecha_Fin_IMPREVISTAS': row1['Fecha Fin'],
                'Fecha_Fin_EVENTLIST': row2['Fecha Fin'],
                'Diferencia_Inicio_Minutos': round(diff_inicio, 2),
                'Diferencia_Fin_Minutos': round(diff_fin, 2),
                'Diferencia_Observable': diferencia_observable,
                'Referencia': referencia,
                'Estado_Coincidencia': 'Coincidencia Aproximada'
            })

if resultados_aproximados:
    df_aproximados = pd.DataFrame(resultados_aproximados)
    print(f"✓ Coincidencias aproximadas encontradas: {len(df_aproximados)} registros")
    
    # Exportar todas las coincidencias aproximadas
    exportar_excel(df_aproximados, '02_Coincidencias_Aproximadas_Completas.xlsx')
    
    # Exportar solo las observables
    df_observables = df_aproximados[df_aproximados['Diferencia_Observable'] == 'Observable']
    if len(df_observables) > 0:
        exportar_excel(df_observables, '03_Coincidencias_Observables.xlsx')
    
    # Exportar solo las permitidas
    df_permitidas = df_aproximados[df_aproximados['Diferencia_Observable'] == 'Permitido']
    if len(df_permitidas) > 0:
        exportar_excel(df_permitidas, '04_Coincidencias_Permitidas.xlsx')
    
    # Resumen de coincidencias aproximadas
    resumen_aproximadas = pd.DataFrame({
        'Tipo': ['Coincidencias Aproximadas', 'Observables', 'Permitidas'],
        'Cantidad_Registros': [
            len(df_aproximados),
            len(df_observables),
            len(df_permitidas)
        ]
    })
    
else:
    print("✗ No se encontraron coincidencias aproximadas")
    df_aproximados = pd.DataFrame()
    resumen_aproximadas = pd.DataFrame()

# 3. REGISTROS SIN COINCIDENCIA
print("\n" + "="*60)
print("3. IDENTIFICANDO REGISTROS SIN COINCIDENCIA")
print("="*60)

# Registros en df1 sin coincidencia en df2
lmt_con_coincidencia = set(df_exactas['LMT']).union(set([r['LMT'] for r in resultados_aproximados]))
df_sin_coincidencia = df1[~df1['LMT'].isin(lmt_con_coincidencia)]

if len(df_sin_coincidencia) > 0:
    print(f"✓ Registros sin coincidencia: {len(df_sin_coincidencia)}")
    exportar_excel(df_sin_coincidencia, '05_Registros_Sin_Coincidencia.xlsx')
else:
    print("✓ Todos los registros tienen al menos una coincidencia")
    df_sin_coincidencia = pd.DataFrame()

# 4. REPORTE FINAL CONSOLIDADO
print("\n" + "="*60)
print("4. GENERANDO REPORTE FINAL")
print("="*60)

# Crear reporte consolidado
reporte_final = pd.DataFrame({
    'Categoria': [
        'Total registros IMPREVISTAS CC',
        'Total registros EventList',
        'Coincidencias Exactas',
        'Coincidencias Aproximadas',
        ' - Observables (>5 min diferencia)',
        ' - Permitidas (≤5 min diferencia)',
        'Registros sin coincidencia'
    ],
    'Cantidad': [
        len(df1),
        len(df2),
        len(df_exactas),
        len(df_aproximados) if resultados_aproximados else 0,
        len(df_observables) if resultados_aproximados else 0,
        len(df_permitidas) if resultados_aproximados else 0,
        len(df_sin_coincidencia)
    ]
})

# Exportar reporte final
exportar_excel(reporte_final, '06_Reporte_Final_Consolidado.xlsx')

# 5. DETALLE POR LMT
print("\n" + "="*60)
print("5. GENERANDO ANÁLISIS POR LMT")
print("="*60)

# Combinar todos los resultados para análisis por LMT
todos_resultados = []
if len(df_exactas) > 0:
    for _, row in df_exactas.iterrows():
        todos_resultados.append({
            'LMT': row['LMT'],
            'Tipo_Coincidencia': 'Exacta',
            'Diferencia_Observable': 'Permitido',
            'Referencia': row.get('Referencia', 'N/A'),
            'Fecha_Inicio': row['Fecha inicio-hora']
        })

if resultados_aproximados:
    for resultado in resultados_aproximados:
        todos_resultados.append({
            'LMT': resultado['LMT'],
            'Tipo_Coincidencia': 'Aproximada',
            'Diferencia_Observable': resultado['Diferencia_Observable'],
            'Referencia': resultado['Referencia'],
            'Fecha_Inicio': resultado['Fecha_Inicio_IMPREVISTAS']
        })

if todos_resultados:
    df_analisis_lmt = pd.DataFrame(todos_resultados)
    
    # Resumen por LMT
    resumen_lmt = df_analisis_lmt.groupby('LMT').agg({
        'Tipo_Coincidencia': 'count',
        'Diferencia_Observable': lambda x: (x == 'Observable').sum()
    }).reset_index()
    
    resumen_lmt.columns = ['LMT', 'Total_Coincidencias', 'Coincidencias_Observables']
    resumen_lmt['Porcentaje_Observables'] = round((resumen_lmt['Coincidencias_Observables'] / resumen_lmt['Total_Coincidencias']) * 100, 2)
    
    exportar_excel(resumen_lmt, '07_Analisis_Por_LMT.xlsx')
    exportar_excel(df_analisis_lmt, '08_Detalle_Completo_Coincidencias.xlsx')

# RESUMEN EJECUTIVO EN CONSOLA
print("\n" + "="*80)
print("🎯 RESUMEN EJECUTIVO DEL CRUCE")
print("="*80)
print(f"📊 Total registros procesados:")
print(f"   • IMPREVISTAS CC: {len(df1)} registros")
print(f"   • EventList: {len(df2)} registros")
print(f"   • LMT únicos en IMPREVISTAS CC: {df1['LMT'].nunique()}")
print(f"   • LMT únicos en EventList: {df2['LMT'].nunique()}")

print(f"\n✅ Resultados del cruce:")
print(f"   • Coincidencias exactas: {len(df_exactas)}")
if resultados_aproximados:
    print(f"   • Coincidencias aproximadas: {len(df_aproximados)}")
    print(f"     - Observables: {len(df_observables) if 'df_observables' in locals() else 0}")
    print(f"     - Permitidas: {len(df_permitidas) if 'df_permitidas' in locals() else 0}")
print(f"   • Sin coincidencia: {len(df_sin_coincidencia)}")

print(f"\n💾 Archivos exportados en: {output_dir}/")
archivos_exportados = [f for f in os.listdir(output_dir) if f.endswith('.xlsx')]
for archivo in sorted(archivos_exportados):
    print(f"   📄 {archivo}")

print(f"\n📍 Ubicación completa: {os.path.abspath(output_dir)}")
print("="*80)

Columnas en df1: ['DIA', 'GEMAS', 'REF CC', 'NT', 'UOT', 'Zona', 'CFT', 'Subestación', 'LMT', 'Estado', 'Fecha inicio-hora', 'Fecha Fin', 'Tipo Evento', 'Causa', 'Ticket', 'Cl. Afec.', 'G.res', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39']
Columnas en df2: ['Referencia', 'NT', 'UOT', 'Zona', 'CFT', 'Subestación', 'LMT', 'CD', 'QBT', 'LBT', 'PCR', 'Duración', 'Estado', 'Estado DOC', 'Fecha inicio/hora', 'Fecha Fin', 'Tipo Evento', 'Causa', 'Ticket', 'G. ori.', 'Cl. Afec.', 'Cl. Sing.', 'G.res', 'Estado Cadis', 'Error Cadis', 'N.Descargo', 'Sistema Origen', 'Recursos', 'Cadis Data', 'Codigo Uot', 'Codigo Zona']


,DIA,GEMAS,REF CC,NT,UOT,Zona,CFT,Subestación,LMT,Estado,...,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39
0,1,B074048,B074048,MT,Norte Chico,NaN,HUAURA,Huacho,H-10,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,B074051,B074051,MT,Panamericana,NaN,CALLAO,Oquendo,O-09,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,B074054,B074054,MT,Panamericana,NaN,CALLAO,Santa Marina,F-18,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,B074055,B074055,MT,Panamericana,NaN,CARABAYLLO,Huarangal,NZ-03,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,B074056,B074056,MT,Norte Chico,NaN,CHANCAY,Chancay,CY-05,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
341,20,PM-TMP-260121011337,PM-TMP-260121011337,MT,Norte Chico,27 DE NOVIEMBRE,Acos Hoyos,NaN,AC-01,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
342,20,B074998,B074998,MT,Panamericana,COMAS,Infantas,NaN,I-22,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EVENTO PERDIDO EN EL PROCESO MERGE,NaN
343,20,PM-TMP-260120203446,PM-TMP-260120203446,MT,Panamericana,CALLAO,Barsi,NaN,K-16,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
344,20,PM-TMP-260121011613,PM-TMP-260121011613,MT,Norte Chico,27 DE NOVIEMBRE,Acos Hoyos,NaN,AC-01,CERRADO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Referencia,NT,UOT,Zona,CFT,Subestación,LMT,CD,QBT,LBT,...,Cl. Sing.,G.res,Estado Cadis,Error Cadis,N.Descargo,Sistema Origen,Recursos,Cadis Data,Codigo Uot,Codigo Zona
0,B075061,MT,UO NORTE CHICO,ZO NORTE CHICO,27 DE NOVIEMBRE,HOYOS-ACOS,AC-01,NaN,NaN,NaN,...,624.0,P,NaN,NaN,NaN,NaN,NaN,NaN,EE1CU1,EE1C
1,B075060-01,MT,UO COLONIAL,ZO COLONIAL,LIMA CERCADO,TACNA,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EE1AU1,EE1A
2,B075059,MT,UO COLONIAL,ZO COLONIAL,LIMA CERCADO,MALVINAS,MS-06,00437S,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EE1AU1,EE1A
3,B075058,MT,UO COLONIAL,ZO COLONIAL,LIMA CERCADO,MALVINAS,MS-06,00050S,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EE1AU1,EE1A
4,B075057,MT,UO COLONIAL,ZO COLONIAL,SAN JUAN DE LURIGANCHO,CTO_GRANDE,CG-02,NaN,NaN,NaN,...,8.0,P,NaN,NaN,NaN,NaN,NaN,NaN,EE1AU1,EE1A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887,B074051,MT,UO PANAMERICANA,ZO PANAMERICANA,CALLAO,OQUENDO,O-09,NaN,NaN,NaN,...,1924.0,P,NaN,NaN,NaN,NaN,NaN,NaN,EE1BU1,EE1B
888,B074050,MT,UO NORTE CHICO,ZO NORTE CHICO,HUAURA,HUACHO,H-01,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EE1CU1,EE1C
889,B074049,MT,UO NORTE CHICO,ZO NORTE CHICO,HUAURA,HUACHO,H-01,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EE1CU1,EE1C
890,B074048,MT,UO NORTE CHICO,ZO NORTE CHICO,HUAURA,HUACHO,H-01,22172A,NaN,NaN,...,5711.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EE1CU1,EE1C



1. BUSCANDO COINCIDENCIAS EXACTAS
✓ Coincidencias exactas encontradas: 214 registros
✗ Error exportando 01_Coincidencias_Exactas.xlsx: '[' is not a valid column name. Column names are from A to ZZZ

2. BUSCANDO COINCIDENCIAS APROXIMADAS
✓ Coincidencias aproximadas encontradas: 462 registros
✓ Archivo exportado: 02_Coincidencias_Aproximadas_Completas.xlsx
✓ Archivo exportado: 03_Coincidencias_Observables.xlsx
✓ Archivo exportado: 04_Coincidencias_Permitidas.xlsx

3. IDENTIFICANDO REGISTROS SIN COINCIDENCIA
✓ Registros sin coincidencia: 14
✗ Error exportando 05_Registros_Sin_Coincidencia.xlsx: '[' is not a valid column name. Column names are from A to ZZZ

4. GENERANDO REPORTE FINAL
✓ Archivo exportado: 06_Reporte_Final_Consolidado.xlsx

5. GENERANDO ANÁLISIS POR LMT
✓ Archivo exportado: 07_Analisis_Por_LMT.xlsx
✓ Archivo exportado: 08_Detalle_Completo_Coincidencias.xlsx

🎯 RESUMEN EJECUTIVO DEL CRUCE
📊 Total registros procesados:
   • IMPREVISTAS CC: 346 registros
   • EventList: 892 r